# 03b — Free-text: Deploy and Monitor

Deploy the complete model from `02c` (or the ready-made sample model in
`sample_data/models/`, if you haven't run it) and keep scoring new data with the
`amygda-risk` monitor, on the sample data split into 13 weekly files (Jan–Mar 2024).

Free-text models classify each new log against the model's keywords, so a call takes longer than for fixed-log, and at most 5,000 rows go in one call (the monitor splits larger files automatically). This sample model leaves about 60% of logs unmatched (routine entries with no subsystem), the same as in its Step 12 run; the monitor only warns when that share rises.

See `03_deploy_and_monitor_guide.ipynb` for what each step does and why.

## Setup

In [ ]:
import glob, json, os, shutil, webbrowser
from pathlib import Path

import pandas as pd

# Installing the SDK first time:
#   !pip install git+https://github.com/Amygda/amygda_ops_risk_score_sdk.git
# Already installed, please update to the latest version:
#   !pip install --upgrade git+https://github.com/Amygda/amygda_ops_risk_score_sdk.git

import amygda_ops_risk_score
from amygda_ops_risk_score import OpsRiskClient, helpers
from amygda_ops_risk_score.monitor import Monitor, MonitorError

print("amygda_ops_risk_score version:", amygda_ops_risk_score.__version__)

In [ ]:
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file

API_KEY = os.environ.get("AMYGDA_SDK_API_KEY") or "paste_your_api_key_here"   # ← paste your key here

# The monitor and the `amygda-risk` command read the key from AMYGDA_API_KEY.
# It is never written to the monitor folder.
os.environ["AMYGDA_API_KEY"] = API_KEY

client = OpsRiskClient()
client.wait_until_ready()
print("API is ready.")

In [ ]:
# ── Paths — edit once ────────────────────────────────────────────────────────
# Yours from 02c if you ran it, otherwise the ready-made sample model.
MODEL_ZIPS = sorted(glob.glob("outputs/free_text/complete_model_*.zip"), key=os.path.getmtime)
MODEL      = MODEL_ZIPS[-1] if MODEL_ZIPS else "sample_data/models/free_text_complete_model.zip"
TRAINING   = "sample_data/free_text_training.csv"     # earlier logs, used as history
BATCHES    = Path("sample_data/batches/free_text")    # week_01.csv … week_13.csv
MON_DIR    = Path("outputs/free_text_monitor")
MON_NAME   = "example-free-text"

week = {int(p.stem[-2:]): p for p in sorted(BATCHES.glob("week_*.csv"))}
print("Model:", MODEL)

# Start clean: remove the folder and deployment from an earlier run of this example
for d in client.list_deployments(API_KEY):
    if d["name"] == MON_NAME:
        client.delete_deployment(API_KEY, d["deployment_id"])
shutil.rmtree(MON_DIR, ignore_errors=True)

## Deploy and create the monitor

Deploying stores a small scoring bundle on Amygda's servers (no log rows, asset IDs or
timestamps). Read the wording below; `--accept-storage` accepts it.

In [ ]:
print(client.deployment_storage_policy())

In [ ]:
!amygda-risk init {MON_DIR} --model {MODEL} --name {MON_NAME} --accept-storage \
    --history {TRAINING} --fleet-name "Rolling stock HVAC"

## Run the first weeks

`--input` takes one or more files. The period holding each run's newest log is marked
provisional until the next run completes it.

In [ ]:
!amygda-risk run {MON_DIR} --input {week[1]}

In [ ]:
for n in range(2, 8):
    !amygda-risk run {MON_DIR} --input {week[n]} --no-dashboard

## The rest through the inbox

A scheduled job would run `amygda-risk run <folder>` with no `--input`: it scores everything
in `inbox/` and moves the files to `inbox/processed/`.

In [ ]:
for n in range(8, 14):
    shutil.copy(week[n], MON_DIR / "inbox" / week[n].name)
!amygda-risk run {MON_DIR}

## Status, runs and scores

In [ ]:
!amygda-risk status {MON_DIR}

In [ ]:
mon = Monitor(MON_DIR)
cols = ["run_id", "status", "rows_new", "periods_upserted", "unmatched_share",
        "provisional_periods", "warnings"]
pd.DataFrame(mon.runs()).reindex(columns=cols)

In [ ]:
scores = mon.scores
print(f"{len(scores):,} periods for {scores['asset_id'].nunique()} assets, "
      f"{scores['period_start'].min()} to {scores['period_start'].max()}")
(scores.groupby("asset_id")["operational_risk"].agg(["mean", "max"]).round(1)
       .sort_values("mean", ascending=False))

## Dashboard

In [ ]:
dash = (MON_DIR / "dashboard" / "index.html").resolve()
print(dash)
webbrowser.open(dash.as_uri())

## One-off scoring

Score a file with the same model without changing the monitor.

In [ ]:
!amygda-risk score {MON_DIR} --input {week[3]} --output outputs/{MON_NAME}_oneoff

## Clean up (optional)

Deletes the deployment and the monitor folder.

In [ ]:
DELETE = False   # ← set True to delete
if DELETE:
    client.delete_deployment(API_KEY, mon.deployment["deployment_id"])
    shutil.rmtree(MON_DIR, ignore_errors=True)
    shutil.rmtree(f"outputs/{MON_NAME}_oneoff", ignore_errors=True)